# Example 1.2.2 / 3: Always Up to Date

Companion notebook for **1.2.2 Computed Parameters** in the AMPL lab Documentation, section "Always Up to Date". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Always Up to Date

AMPL keeps every `=` definition current. When data it depends on changes, the computed value
changes too. From Python, assigning a new value with `ampl.param["frac"] = 0.2` is the
change:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param frac > 0;
param market {PROD} >= 0;
param mininv {p in PROD} = frac * market[p];   # minimum inventory

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["frac"] = 0.1
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
print(ampl.param["mininv"].to_pandas())
ampl.param["frac"] = 0.2
print(ampl.param["mininv"].to_pandas())

       mininv
bands     600
coils     400
       mininv
bands    1200
coils     800


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The `let` command changes the value:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param frac := 0.1;
param market := bands 6000  coils 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display mininv;
let frac := 0.2;
display mininv;

mininv [*] :=
bands  600
coils  400
;

mininv [*] :=
bands  1200
coils   800
;

